## This is an example showcasing the conversion of angle-mapped ARPES data into momentum space using pyarpes
The band dispersion is loaded from an hdf5 format and converted into the [NeXus format](https://manual.nexusformat.org/classes/contributed_definitions/NXmpes_arpes.html#nxmpes_arpes) using the [Nomad Parser Nexus](https://github.com/nomad-coe/nomad-parser-nexus).
Subsequently, the converted NeXus file is imported into pyarpes, and the Fermi Surface is converted into momentum space.

Import all required modules

In [ ]:
#%load_ext autoreload
#%autoreload 2
from pynxtools.dataconverter.convert import convert
from arpes.io import load_data
from arpes.endstations.plugin.nexus import NeXusEndstation
from arpes.utilities.conversion import convert_to_kspace
import numpy as np

Convert the file into the NeXus format using the pynxtools data converter

In [ ]:
convert(nxdl="NXmpes_arpes", reader="mpes", output="FS_map_TbTe3.nxs", input_file=("FS_map_TbTe3.h5", "NXmpes_phoibos.json"))

Load into pyarpes with the NeXusEndstation loader

In [ ]:
FS_map_TbTe3 = load_data("FS_map_TbTe3.nxs", location=NeXusEndstation)

shift energy axis by photon energy

In [ ]:
FS_map_TbTe3_shifted = FS_map_TbTe3
FS_map_TbTe3_shifted['eV'] = FS_map_TbTe3_shifted['eV'] - FS_map_TbTe3.attrs["hv"].magnitude

Inspect resulting dataset

In [ ]:
FS_map_TbTe3_shifted

Extract Fermi surface

In [ ]:
# map 1 eV below EF
fsmap = FS_map_TbTe3_shifted.S.generic_fermi_surface(0)

Convert into momentum space

In [ ]:

converted = convert_to_kspace(
    fsmap, # just convert the Fermi surface
    kx=np.linspace(-.5, .5, 200),   # along -2.5 <= kx < 1.5 (inv ang.)
                                      #  with 400 steps
    ky=np.linspace(-1.1, 1.1, 400),       # as above, with -2 <= ky < 2
)

In [ ]:
%matplotlib inline
converted.T.plot()